# 03 Baseline Model
Train and evaluate a logistic regression baseline using temporal split.

In [3]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path('..').resolve()))
from src.model import build_baseline_model, train_pipeline, predict_with_scores
from src.preprocessing import add_hit_label, make_preprocessor, temporal_split
from src.evaluation import classification_metrics

df = pd.read_csv('../data/raw/spotify.csv')
df.columns = [c.lower() for c in df.columns]

# Use existing label if present; otherwise derive from popularity
if 'hit' not in df.columns:
    popularity_col = 'popularity' if 'popularity' in df.columns else 'track_popularity'
    df = add_hit_label(df, popularity_col=popularity_col, threshold=70)

In [4]:
target_col = 'hit'
time_col = 'year' if 'year' in df.columns else None

feature_df = df.copy()
if time_col is None:
    feature_df['__time_proxy__'] = range(len(feature_df))
    time_col = '__time_proxy__'

x_train, x_test, y_train, y_test = temporal_split(feature_df, time_col=time_col, target_col=target_col)
num_cols = x_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in x_train.columns if c not in num_cols]
preprocessor = make_preprocessor(num_cols, cat_cols)
model = train_pipeline(preprocessor, build_baseline_model(), x_train, y_train)
y_pred, y_score = predict_with_scores(model, x_test)
classification_metrics(y_test, y_pred, y_score)

{'accuracy': 0.9842319430315362,
 'precision_macro': 0.9574175824175823,
 'recall_macro': 0.9905082669932639,
 'f1_macro': 0.9729704917741968,
 'roc_auc': 0.9997574631173325}